In [1]:
import geopandas as gpd
import pandas as pd
import os
import dotenv
import requests
import branca.colormap as cm
import boto3
from pathlib import Path
import re

dotenv.load_dotenv()

True

In [2]:
year = 2024
geo = "county subdivision"
variables = ["NAME", "B01003_001E"]
census_api_key = os.getenv("CENSUS_API_KEY")
state_id=44

variable_string = ",".join(variables)
print(variable_string)
url = f"https://api.census.gov/data/{year}/acs/acs5?get={variable_string}&key={census_api_key}&for={geo}:*&in=state:{state_id}"

res = requests.get(url)
census_pop_data = res.json()

NAME,B01003_001E


In [3]:
town_pop_df = pd.DataFrame(census_pop_data[1:], columns=census_pop_data[0])
town_pop_df = town_pop_df.rename({"B01003_001E": "Total Population Estimate"}, axis='columns')
town_pop_df['Total Population Estimate'] = town_pop_df['Total Population Estimate'].astype(int)
town_pop_df['GEOID'] = town_pop_df["state"] + town_pop_df["county"] + town_pop_df["county subdivision"]
town_pop_df.head()

,NAME,Total Population Estimate,state,county,county subdivision,GEOID
0,"Barrington town, Bristol County, Rhode Island",17126,44,001,05140,4400105140
1,"Bristol town, Bristol County, Rhode Island",22220,44,001,09280,4400109280
2,"Warren town, Bristol County, Rhode Island",11144,44,001,73760,4400173760
3,"Coventry town, Kent County, Rhode Island",35907,44,003,18640,4400318640
4,"East Greenwich town, Kent County, Rhode Island",14554,44,003,22240,4400322240


In [4]:
cart_url = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_44_cousub_500k.zip"
cart_save_path = "../data/ri_cart_2024.zip"
if not os.path.isfile(cart_save_path):
    r = requests.get(cart_url, stream=True)
    with open(cart_save_path, 'wb') as fd:
        for chunk in r.iter_content(chunk_size=512):
            fd.write(chunk)
town_geos = gpd.read_file(cart_save_path)
town_geos.head()

,STATEFP,COUNTYFP,COUSUBFP,COUSUBNS,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,44,003,77720,01220059,0600000US4400377720,4400377720,West Greenwich,West Greenwich town,RI,Kent County,Rhode Island,43,130269418,2598306,"POLYGON ((-71.78936 41.59691, -71.78786 41.634..."
1,44,007,14140,01220067,0600000US4400714140,4400714140,Central Falls,Central Falls city,RI,Providence County,Rhode Island,25,3092443,208466,"POLYGON ((-71.40706 41.89191, -71.40469 41.894..."
2,44,007,64220,01220077,0600000US4400764220,4400764220,Scituate,Scituate town,RI,Providence County,Rhode Island,43,124665548,17111431,"POLYGON ((-71.69101 41.85646, -71.67681 41.856..."
3,44,007,27460,01220072,0600000US4400727460,4400727460,Foster,Foster town,RI,Providence County,Rhode Island,43,131446005,2835602,"POLYGON ((-71.79469 41.85441, -71.75822 41.855..."
4,44,007,11800,01220081,0600000US4400711800,4400711800,Burrillville,Burrillville town,RI,Providence County,Rhode Island,43,142233929,5391503,"POLYGON ((-71.79924 42.00806, -71.76601 42.009..."


In [5]:
town_pop_gdf = town_geos.merge(town_pop_df, on="GEOID")
town_pop_gdf = town_pop_gdf.rename({"NAME_x": "Town Name", "NAMELSADCO": "County Name"}, axis='columns')
town_pop_gdf = town_pop_gdf[["Town Name", "County Name", "Total Population Estimate", "geometry"]]
town_pop_gdf.head()

,Town Name,County Name,Total Population Estimate,geometry
0,West Greenwich,Kent County,6647,"POLYGON ((-71.78936 41.59691, -71.78786 41.634..."
1,Central Falls,Providence County,22701,"POLYGON ((-71.40706 41.89191, -71.40469 41.894..."
2,Scituate,Providence County,10517,"POLYGON ((-71.69101 41.85646, -71.67681 41.856..."
3,Foster,Providence County,4523,"POLYGON ((-71.79469 41.85441, -71.75822 41.855..."
4,Burrillville,Providence County,16427,"POLYGON ((-71.79924 42.00806, -71.76601 42.009..."


In [ ]:
vmin = 0
vmax = 200000
vmid = (vmax+vmin) / 2
colormap = cm.linear.GnBu_09.scale(vmin, vmax)
colormap.caption = "Population Estimate 2024"
colormap.tick_labels = [vmin, vmid, vmax]


town_pop_map = town_pop_gdf.explore(
    column="Total Population Estimate",
    tiles=None,
    legend=True,
    vmin=vmin,
    vmax=vmax,
    cmap=colormap,
    style_kwds=dict(color="black"),
)

map_export_path = "../maps/RITownPopulation2024.html"
town_pop_map.save(map_export_path)

In [17]:
town_pop_map.show_in_browser()

Your map should have been opened in your browser automatically.
Press ctrl+c to return.


In [8]:
session = boto3.Session(profile_name="dev", region_name="us-east-2")
s3 = session.resource('s3')
s3.meta.client.upload_file(Filename=map_export_path, Bucket='lrm-map-file-exports', Key="RITownPopulation2024.html", ExtraArgs={'ContentType': "text/html", 'ACL': 'public-read'})

TokenRetrievalError: Error when retrieving token from sso: Token has expired and refresh failed